# 特征提取

输出 `features/<model>.npz`（`image_ids`、`features`）。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from fmspace.data import load_metadata
from fmspace.extract import extract, load_features, save_features
from fmspace.models import MODELS, load_encoder

DATA_DIR = Path(r"C:\M1\master paper\DataSet\DeepDRiD")
FEATURE_DIR = Path("../features")
FEATURE_DIR.mkdir(exist_ok=True)

In [ ]:
metadata = load_metadata(DATA_DIR)
metadata

## 试跑

In [ ]:
X = extract(load_encoder("siglip"), metadata["path"][:8])
X.shape

## 全量提取

In [ ]:
for name in MODELS:
    X = extract(load_encoder(name), metadata["path"])
    save_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"], X)

## 和旧特征比较

L2 归一化后逐张算余弦相似度。

In [ ]:
OLD_DIR = Path(r"C:\path\to\SAMIT\features\DeepDRiD")


def l2(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)


cos = pd.DataFrame(
    {
        name: (
            l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"]))
            * l2(load_features(OLD_DIR / f"{name}.npz", metadata["image_id"]))
        ).sum(axis=1)
        for name in MODELS
    }
)
summary = cos.describe().T[["min", "50%"]]
summary.to_csv("../results/feature_check.csv")
summary